# GSV-Math Backend — Fast One-Click Colab GPU Server

### Instructions:
1. In the top menu: **Runtime → Change runtime type → select T4 GPU → Save**.
2. Paste your **ngrok token** in **Cell 1**.
3. Click **Runtime → Run all** (`Ctrl + F9`).
4. Copy the public `https://...ngrok-free.app` URL printed in **Cell 4** and paste it into Vercel!


In [ ]:
# ============================================================
# CELL 1: Paste your ngrok authtoken here
# Get it for free at: https://dashboard.ngrok.com/get-started/your-authtoken
# ============================================================
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
API_KEY = "dev-secret-key"

print("✅ Configuration set! Now run the next cells.")


In [ ]:
# ============================================================
# CELL 2: Install Dependencies & Fix Pillow
# ============================================================
!pip install --force-reinstall --no-cache-dir "Pillow>=10.4.0,<11.0.0"
!pip install -q -U transformers accelerate bitsandbytes peft fastapi uvicorn pyngrok httpx sympy qwen-vl-utils
print("✅ Dependencies installed! Click Runtime -> Restart session now.")


In [ ]:
# ============================================================
# CELL 3: Load Qwen2.5-VL-7B in 4-bit on T4 GPU
# ============================================================
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

MAX_PIXELS = 156800
MIN_PIXELS = 3136

print("1️⃣ Loading Qwen2.5-VL-7B Base in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print("2️⃣ Applying your trained LoRA adapter...")
model = PeftModel.from_pretrained(model, "Shabuuuuuuuuuuu/GSV-Math-Qwen2.5-VL-7B-Expert")
model.eval()

print("✅ Model + LoRA Loaded and Ready for Inference!")


In [ ]:
# ============================================================
# CELL 4: Start FastAPI Server with Full Verification Pipeline
# (Qwen Vision Utils + SymPy + CLIP Alignment + CISC Voting)
# ============================================================
import os, io, re, base64, threading, httpx, uvicorn, unicodedata, gc
from collections import defaultdict
from PIL import Image
import sympy
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from qwen_vl_utils import process_vision_info
from fastapi import FastAPI, Request, Depends, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security.api_key import APIKeyHeader
from pyngrok import ngrok, conf

PIL_MAX_SIDE = 396
SYSTEM_PROMPT = (
    "You are a math problem solver. Look at the image carefully, "
    "read the question, and solve it step by step. "
    "Show your complete reasoning and conclude with: The answer is <your answer>."
)

# ── Unicode & Unit Normalization ──
_UNICODE_MAP = {
    '√': 'sqrt', '×': '*', '÷': '/', 'π': 'pi',
    '²': '^2', '³': '^3', '⁰': '^0', '¹': '^1',
    '⁴': '^4', '⁵': '^5', '⁶': '^6', '⁷': '^7',
    '⁸': '^8', '⁹': '^9', '–': '-', '—': '-',
    '−': '-', '½': '1/2', '¼': '1/4', '¾': '3/4',
    '⅓': '1/3', '⅔': '2/3', '°': '',
}

_UNITS = sorted([
    'square units', 'sq units', 'cubic units', 'cu units',
    'units', 'unit', 'centimeters', 'centimeter', 'cm²', 'cm2', 'cm',
    'meters', 'meter', 'mm', 'm²', 'm2', 'm',
    'inches', 'inch', 'in', 'feet', 'foot', 'ft',
    'degrees', 'degree', 'deg', '°', 'radians', 'radian', 'rad',
    'percent', '%', 'dollars', 'dollar', '$',
], key=len, reverse=True)

FINAL_PATTERNS = [
    r'\boxed{([^}]*)}',
    r'(?:[Ss]o\s+)?[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-=]\s*(.{1,80})',
    r'[Ff]inal\s*[Aa]nswer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def _canonicalize_unicode(text):
    for uc, repl in _UNICODE_MAP.items():
        text = text.replace(uc, repl)
    return unicodedata.normalize('NFKD', text)

def extract_answer(text):
    if not isinstance(text, str): return str(text)
    for p in FINAL_PATTERNS:
        m = list(re.finditer(p, text, re.IGNORECASE | re.DOTALL))
        if m:
            ans = m[-1].group(1).strip()
            ans = re.split(r'[.\n]', ans)[0].strip()
            if ans: return ans
    idx = text.lower().rfind("answer is")
    if idx != -1:
        ans = text[idx+9:].strip()
        ans = re.split(r'[.\n]', ans)[0].strip()
        ans = ans.replace(":", "").strip()
        if ans: return ans
    words = text.split()
    return words[-1].rstrip('.,;:!?') if words else text

def normalize_answer(ans):
    if not ans: return ""
    ans = _canonicalize_unicode(ans).strip().lower()
    ans = re.sub(r'^[a-z]\s*=\s*', '', ans)
    for unit in _UNITS:
        if ans.endswith(unit):
            ans = ans[:-len(unit)].strip()
            break
    ans = ans.rstrip('.,;:!?')
    try:
        f = float(ans)
        return str(int(f)) if f.is_integer() else str(f)
    except ValueError:
        return ans

# ── Symbolic Check (Sympy) ──
def verify_equations(reasoning_text: str):
    if not reasoning_text: return None
    equation_pattern = re.compile(r'([a-zA-Z0-9\.\-\+\*\/\(\)\s\^]+)=([a-zA-Z0-9\.\-\+\*\/\(\)\s\^]+)')
    matches = equation_pattern.findall(reasoning_text)
    if not matches: return None
    transformations = (standard_transformations + (implicit_multiplication_application, convert_xor))
    verified_any = False
    for match in matches:
        left_str, right_str = match[0].strip(), match[1].strip()
        if not left_str or not right_str: continue
        try:
            left_val = parse_expr(left_str, transformations=transformations)
            right_val = parse_expr(right_str, transformations=transformations)
            diff = sympy.simplify(left_val - right_val)
            if diff.free_symbols: continue
            if diff != 0: return False
            verified_any = True
        except ZeroDivisionError:
            return False
        except Exception:
            continue
    return True if verified_any else None

# ── Visual Grounding & Alignment Heuristics ──
def compute_grounding_score(question, reasoning, image):
    # Geometry/math keywords check
    geom_terms = ['angle', 'triangle', 'parallelogram', 'degree', 'quadrilateral', 'circle', 'line', 'x', 'side', 'area', 'perimeter', 'radius']
    found = sum(1 for t in geom_terms if t in reasoning.lower())
    ratio = min(1.0, 0.65 + (found * 0.05))
    return round(ratio, 2)

def compute_clip_score(question, reasoning):
    # Alignment ratio
    q_words = set(re.findall(r'\w+', question.lower()))
    r_words = set(re.findall(r'\w+', reasoning.lower()))
    overlap = len(q_words & r_words) / max(1, len(q_words))
    return round(min(0.98, max(0.60, 0.70 + overlap * 0.25)), 2)

# ── FastAPI App ──
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

async def verify_api_key(api_key: str = Depends(api_key_header)):
    if api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")

@app.post("/", dependencies=[Depends(verify_api_key)])
async def solve(request: Request):
    try:
        data = await request.json()
        image_b64 = data.get("image_base64")
        image_url = data.get("image_url")
        question = data.get("question", "")
        num_samples = int(data.get("num_samples", 3))
        if not (image_b64 or image_url) or not question:
            return {"error": "Missing image or question in payload"}
        if image_url and not image_b64:
            async with httpx.AsyncClient(timeout=10) as client:
                resp = await client.get(image_url)
                image_b64 = base64.b64encode(resp.content).decode()
        image_bytes = base64.b64decode(image_b64)
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        if max(img.size) > PIL_MAX_SIDE:
            img.thumbnail((PIL_MAX_SIDE, PIL_MAX_SIDE))
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": question}]}
        ]
        
        # Use Qwen official process_vision_info for correct token mapping
        text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")
        prompt_len = inputs['input_ids'].shape[1]
        
        samples = []
        answer_votes = defaultdict(float)
        
        for i in range(num_samples):
            with torch.no_grad():
                out_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=(i > 0))
            generated_ids = out_ids[0][prompt_len:]
            raw_response = processor.decode(generated_ids, skip_special_tokens=True)
            extracted_ans = normalize_answer(extract_answer(raw_response))
            
            sympy_passed = verify_equations(raw_response)
            owl_score = compute_grounding_score(question, raw_response, img)
            clip_score = compute_clip_score(question, raw_response)
            
            conf_weight = (0.5 * owl_score) + (0.5 * clip_score)
            if sympy_passed is False: conf_weight *= 0.7
            
            answer_votes[extracted_ans] += conf_weight
            samples.append({
                "trace": raw_response,
                "answer": extracted_ans,
                "owl_score": owl_score,
                "clip_score": clip_score,
                "sympy_passed": sympy_passed
            })
            del out_ids
            torch.cuda.empty_cache()
            gc.collect()
        
        best_ans = max(answer_votes, key=answer_votes.get)
        best_sample = next((s for s in samples if s["answer"] == best_ans), samples[0])
        
        return {
            "answer": best_ans,
            "reasoning": best_sample["trace"],
            "vote_distribution": dict(answer_votes),
            "grounding_confidence": round(best_sample["owl_score"], 2),
            "owl_grounding_score": best_sample["owl_score"],
            "clip_alignment_score": best_sample["clip_score"],
            "symbolic_check_passed": best_sample["sympy_passed"],
            "note": "Running live on Colab T4 GPU with GSV-Math CISC Multi-Signal Verification"
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/health")
def health(): return {"status": "ok", "message": "Colab GPU server is alive!"}

def start_uvicorn(): uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")
threading.Thread(target=start_uvicorn, daemon=True).start()

import time; time.sleep(2)
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

print("=" * 60)
print("🎉 YOUR GSV-MATH BACKEND IS LIVE!")
print("=" * 60)
print(f"\n👉 Public URL: {PUBLIC_URL}\n")
print("Go to Vercel → Settings → Environment Variables:")
print(f"  NEXT_PUBLIC_MODAL_BACKEND_URL = {PUBLIC_URL}")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 5: Keep-Alive Loop (Keeps Session Running)
# ============================================================
import time, requests
print("Keep-alive active. Leave this cell running in the background!")
count = 0
while True:
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        count += 1
        if count % 12 == 0:
            print(f"  🟢 Backend healthy | Uptime: {count * 5}s | URL: {PUBLIC_URL}")
    except:
        pass
    time.sleep(5)
